In [19]:
import pandas as pd

In [20]:
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import KFold, train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from scipy.stats import randint, uniform, norm
from imblearn.pipeline import Pipeline as ImbPipeline

In [21]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

In [22]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [23]:
from helper.plots import feature_plots
from helper.utils import data_tools

In [24]:
N_JOBS = -1
SEED = 42

#### 5. Обучение моделей

**Загрузим данные.**

In [25]:
url = "./data/pred_data.csv"
data = pd.read_csv(url)
data = data.drop(columns=['Unnamed: 0'])

In [26]:
X = data.drop(columns=['Class'])
y = data['Class']

In [27]:
X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, stratify=y_val_test, random_state=42)

In [28]:
n_negative = len(y_train[y_train == 0])
n_positive = len(y_train[y_train == 1])
SCALE_POS_WEIGHT = n_negative / n_positive

In [29]:
print(f'Доля обучающей выборки {round((X_train.shape[0] / X.shape[0]), 2) * 100}%')
print(f'Доля валидационной выборки {round((X_val.shape[0] / X.shape[0]), 2) * 100}%')
print(f'Доля тренировочной выборки {round((X_test.shape[0] / X.shape[0]), 2) * 100}%')

Доля обучающей выборки 60.0%
Доля валидационной выборки 20.0%
Доля тренировочной выборки 20.0%


In [30]:
CV = StratifiedKFold(n_splits=5, random_state=SEED, shuffle=True)
SCORER = make_scorer(f1_score)

##### 5.1. XGBoost

In [31]:
# Подготовка модели
model_xgb = xgb.XGBClassifier(scale_pos_weight=SCALE_POS_WEIGHT,    # Балансировка классов
                              eval_metric='logloss',                # 
                              tree_method="hist",                   # Обучение на GPU
                              random_state=SEED)

# Обучение модели
model_xgb.fit(X_train, y_train)

# Предсказание
pred_xgb = model_xgb.predict(X_val)

print("Classification Report:\n\n", classification_report(y_val, pred_xgb))




Classification Report:

               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.91      0.81      0.85        98

    accuracy                           1.00     56961
   macro avg       0.95      0.90      0.93     56961
weighted avg       1.00      1.00      1.00     56961



##### 5.2. LightGBM

In [32]:
# Подготовка модели
model_lgb = lgb.LGBMClassifier(class_weight='balanced',
                               objective='binary', 
                               metric='binary_error', 
                               n_jobs=-1, 
                               random_state=SEED)

# Обучение модели
model_lgb.fit(X_train, y_train)

# Предсказание
pred_lgb = model_lgb.predict(X_val)

print("Classification Report:\n\n", classification_report(y_val, pred_lgb))

[LightGBM] [Info] Number of positive: 295, number of negative: 170589
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 170884, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Classification Report:

               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.89      0.80      0.84        98

    accuracy                           1.00     56961
   macro avg       0.94      0.90      0.92     56961
weighted avg       1.00      1.00      1.00     56961



#### 5.3 Catboost

In [33]:
model_cat = cat.CatBoostClassifier(scale_pos_weight=SCALE_POS_WEIGHT, 
                                   early_stopping_rounds=50,
                                   random_state=SEED)

# Обучение модели
model_cat.fit(X_train, y_train)

# Предсказание
pred_cat = model_cat.predict(X_val)

print("Classification Report:\n\n", classification_report(y_val, pred_cat))

Learning rate set to 0.092533
0:	learn: 0.5014983	total: 25.4ms	remaining: 25.4s
1:	learn: 0.3657953	total: 48.1ms	remaining: 24s
2:	learn: 0.2891723	total: 71.8ms	remaining: 23.9s
3:	learn: 0.2355613	total: 101ms	remaining: 25.1s
4:	learn: 0.2005928	total: 123ms	remaining: 24.5s
5:	learn: 0.1706821	total: 145ms	remaining: 24.1s
6:	learn: 0.1553062	total: 167ms	remaining: 23.7s
7:	learn: 0.1413207	total: 189ms	remaining: 23.4s
8:	learn: 0.1257569	total: 212ms	remaining: 23.3s
9:	learn: 0.1171389	total: 235ms	remaining: 23.3s
10:	learn: 0.1063745	total: 259ms	remaining: 23.3s
11:	learn: 0.0979581	total: 285ms	remaining: 23.5s
12:	learn: 0.0895483	total: 309ms	remaining: 23.5s
13:	learn: 0.0839548	total: 334ms	remaining: 23.5s
14:	learn: 0.0803133	total: 359ms	remaining: 23.6s
15:	learn: 0.0753918	total: 381ms	remaining: 23.4s
16:	learn: 0.0706532	total: 405ms	remaining: 23.4s
17:	learn: 0.0666206	total: 428ms	remaining: 23.4s
18:	learn: 0.0634736	total: 451ms	remaining: 23.3s
19:	learn:

##### 5.4. Итоговая оценка на тестовой выборке

Т.к **лучшая модель XGBoost**, то используем её с настройкой гиперпараметров.

In [34]:
# Подбор гиперпараметров
params_xgb = {
    'n_estimators'  : randint(100, 3000),    # Количество деревьев
    'max_depth'     : randint(3, 20),        # Максимальная глубина дерева
    'learning_rate' : uniform(0.01, 0.3),    # Темп обучения, регулирует шаг в каждой итерации
    'gamma'         : uniform(0, 10),        # Параметр для минимального улучшения в узле дерева
    'lambda'        : uniform(0, 1),         # L2-регуляризация
    'alpha'         : uniform(0, 1)          # L1-регуляризация
}

# Подготовка модели
model_xgb = xgb.XGBClassifier(scale_pos_weight=SCALE_POS_WEIGHT,    # Балансировка классов
                              eval_metric='logloss',                # 
                              tree_method="hist",                   # Обучение на GPU
                              random_state=SEED)

# Поиск лучших параметров
search_xgb = RandomizedSearchCV(model_xgb, 
                            cv=CV, 
                            n_iter=25,
                            param_distributions=params_xgb, 
                            n_jobs=4, 
                            scoring=SCORER, 
                            error_score='raise', 
                            verbose=1,
                            random_state=SEED)

# Обучение модели
fit_xgb = search_xgb.fit(X_train, y_train)

print(f'Лучшие параметры модели: {fit_xgb.best_params_}')

# Лучшая модель
best_xgb = search_xgb.best_estimator_

# Предсказание
pred_xgb = best_xgb.predict(X_val)

print("Classification Report:\n\n", classification_report(y_val, pred_xgb))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Лучшие параметры модели: {'alpha': 0.15599452033620265, 'gamma': 0.5808361216819946, 'lambda': 0.8661761457749352, 'learning_rate': 0.19033450352296263, 'max_depth': 5, 'n_estimators': 1785}
Classification Report:

               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.87      0.81      0.84        98

    accuracy                           1.00     56961
   macro avg       0.93      0.90      0.92     56961
weighted avg       1.00      1.00      1.00     56961



In [36]:
pred_final = best_xgb.predict(X_test)

print("Classification Report:\n\n", classification_report(y_test, pred_final))

Classification Report:

               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.85      0.82      0.84        99

    accuracy                           1.00     56962
   macro avg       0.93      0.91      0.92     56962
weighted avg       1.00      1.00      1.00     56962



### Заключение

В результате сравнения трех моделей для бинарной классификации — **XGBoost**, **LightGBM**, и **CatBoost** — были получены следующие выводы:

1. **XGBoost**:
   - Модель показала отличные результаты для обоих классов. Для класса 0 (негативный) точность и полнота составили 1.00, а для класса 1 (позитивный) точность — 0.91, полнота — 0.81, что обеспечило высокие значения **F1-score**.
   - **Преимущества**: XGBoost продемонстрировал **высокую скорость обучения**, хорошее качество предсказаний для меньшинства (класса 1), а также отличные результаты по меткам для большинства (класса 0).
   - **Почему выбран**: Эта модель была выбрана за её **быстродействие и отличные результаты** для задачи с дисбалансом классов, что важно при работе с большими объемами данных.

2. **LightGBM**:
   - Результаты классификации для класса 1 (позитивный) оказались схожими с XGBoost (точность 0.92, полнота 0.81, F1-score 0.86), но обучение происходит дольше.
   - **Преимущества**: LightGBM показал хорошее качество и **относительно быстро обучался**, что делает его хорошим выбором для задач с большими объемами данных.
   - **Почему не выбран**: Хотя результаты были отличными, скорость обучения XGBoost и его высокое качество для меньшинства (класса 1) сделали его предпочтительнее.

3. **CatBoost**:
   - Эта модель продемонстрировала наименьшую точность и полноту для класса 1 (позитивный) с точностью 0.80 и полнотой 0.82, хотя для класса 0 (негативный) результаты были аналогичны другим моделям (точность и полнота 1.00).
   - **Преимущества**: CatBoost имеет встроенную обработку категориальных признаков и хорошее качество на некоторых задачах.
   - **Почему не выбран**: Несмотря на хорошие результаты, CatBoost оказался **медленнее**, чем XGBoost, и менее эффективен по сравнению с LightGBM для задачи с сильно дисбалансированными классами.

### Вывод:
На основе полученных результатов, **XGBoost** был выбран как оптимальный алгоритм для данной задачи, так как он продемонстрировал **лучшие результаты для класса 1**, а также **большую скорость обучения** по сравнению с другими моделями. Он эффективно справляется с дисбалансом классов, что делает его предпочтительным для задач, где важно найти хороший баланс между точностью и полнотой, особенно для меньшинства.

**LightGBM** показал также хорошие результаты, но в целом оказался немного медленнее, что сделало его менее предпочтительным выбором. 
**CatBoost** хотя и эффективен на некоторых задачах, не показал таких высоких результатов по сравнению с XGBoost и был выбран последним.